<a href="https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Abstract

This capstone investigates whether search-performance and content signals can help prioritize pages for human content-refresh review. Using an anonymized content dataset, the project evaluates observed search and content signals and develops a ranking-based decision-support workflow. A simple baseline is compared with an ML approach using an honest validation design. The results are interpreted as directional evidence rather than causal proof. The final output is a ranked review queue that helps content teams decide which pages to examine first while keeping the final decision with a human reviewer.

In [40]:
# CAPSTONE SETUP — clone the AIML repository if it is not already available

import os
import subprocess
import pandas as pd

REPO_DIR = "/content/AIML"

if not os.path.exists(REPO_DIR):
    print("AIML repository not found. Cloning repository...")

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/vaishnavikabbe/AIML.git",
            REPO_DIR
        ],
        check=True
    )

    print("Repository cloned successfully.")
else:
    print("AIML repository already exists.")

DATA_PATH = "/content/AIML/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "The repository exists, but the dataset was not found at:\n"
        + DATA_PATH
    )

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Dataset path:", DATA_PATH)

AIML repository already exists.

Dataset loaded successfully!
Rows: 30000
Columns: 44
Dataset path: /content/AIML/data/raw/content_refresh_anonymized.csv


## 1. Question

*The research question and the decision it supports.*

### Research question and decision

My research question is:

**Can search-performance and content signals help prioritize which pages should be reviewed for content refresh first?**

The decision this supports is which pages a content or SEO reviewer should examine first when time and resources for content refresh are limited.

The output is intended as decision-support. It helps organize pages into a review queue, but a human reviewer makes the final decision about whether a page should actually be changed.

In [41]:
# Section 1: Record the research question

research_question = (
    "Can search-performance and content signals help prioritize "
    "which pages should be reviewed for content refresh first?"
)

decision_supported = (
    "Prioritize pages for human content refresh review."
)

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision_supported)

Research question:
Can search-performance and content signals help prioritize which pages should be reviewed for content refresh first?

Decision supported:
Prioritize pages for human content refresh review.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data

This capstone uses the anonymized content-refresh starter dataset available in the project repository.

The dataset contains page-level observations and search/content signals. The unit of analysis is one page/content record.

The analysis uses only fields that are available in the dataset and relevant to the research question. Identifier fields and fields that could create leakage or expose private information are excluded from modeling.

The results are based on the available dataset and its observed measurement window. They should not be interpreted as representing every website, search query, or future search-engine behavior.

In [42]:
# SECTION 2 — DATA

print("Dataset information")
print("--------------------")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(list(df.columns))

print("\nFirst 5 rows:")
display(df.head())

Dataset information
--------------------
Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Yes — that error is because YOUR_BASELINE_VALUE was a placeholder, not actual Python code. Don't use that code.

Since your goal is to get the notebook running without repeatedly getting errors, let's make Section 4 work safely without inventing model numbers.

Replace the entire Section 4 CODE cell with this
# SECTION 4 — RESULTS VS BASELINE

import pandas as pd

print("Results versus baseline")
print("=======================")

# Check whether actual results from previous sections/notebooks exist
baseline_value = globals().get("baseline_metric", None)
model_value = globals().get("model_metric", None)

if baseline_value is not None and model_value is not None:

    results_table = pd.DataFrame({
        "Method": ["Baseline", "ML Model"],
        "Measured metric": [baseline_value, model_value]
    })

    display(results_table)

    if model_value > baseline_value:
        print("The ML model measured higher than the baseline.")
    elif model_value < baseline_value:
        print("The baseline measured higher than the ML model.")
    else:
        print("The ML model and baseline had the same measured value.")

else:

    print("Actual ML-08/ML-09 results are not available in this notebook session.")
    print("No model or baseline numbers have been invented.")
    print()
    print("Section 4 is therefore marked as pending actual results.")

    results_table = pd.DataFrame({
        "Method": ["Baseline", "ML Model"],
        "Measured metric": ["Pending actual result", "Pending actual result"]
    })

    display(results_table)
What you should see

It should run without an error and show:

Method	Measured metric
Baseline	Pending actual result
ML Model	Pending actual result

That's okay temporarily. Do not put fake numbers just to make the table look complete.

But for the final submission

We need the real numbers from your ML-08 and ML-09 notebooks.

Open:

ML-08 → Section 3: Train + compare vs my baseline

and copy the output here.

Then open:

ML-09 → Section 2: My model under an honest split

and copy that output here.

I'll then give you the exact final Section 4 code with your real numbers, so you don't have to guess anything.

Capstone — mirrors your deployed research paper

Open In Colab (image)

This skeleton is yours to fill. Work the sections in order — each one has a one-line hint. Simple words, honest numbers.

Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

svg

[11]

0s

# CAPSTONE SETUP — clone the AIML repository if it is not already available

import os
import subprocess
import pandas as pd

REPO_DIR = "/content/AIML"

if not os.path.exists(REPO_DIR):
    print("AIML repository not found. Cloning repository...")
    
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/vaishnavikabbe/AIML.git",
            REPO_DIR
        ],
        check=True
    )
    
    print("Repository cloned successfully.")
else:
    print("AIML repository already exists.")

DATA_PATH = "/content/AIML/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "The repository exists, but the dataset was not found at:\n"
        + DATA_PATH
    )

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Dataset path:", DATA_PATH)

AIML repository already exists.

Dataset loaded successfully!
Rows: 30000
Columns: 44
Dataset path: /content/AIML/data/raw/content_refresh_anonymized.csv

1. Question

The research question and the decision it supports.

Research question and decision

My research question is:

Can search-performance and content signals help prioritize which pages should be reviewed for content refresh first?

The decision this supports is which pages a content or SEO reviewer should examine first when time and resources for content refresh are limited.

The output is intended as decision-support. It helps organize pages into a review queue, but a human reviewer makes the final decision about whether a page should actually be changed.

svg

[12]

0s

# Section 1: Record the research question

research_question = (
    "Can search-performance and content signals help prioritize "
    "which pages should be reviewed for content refresh first?"
)

decision_supported = (
    "Prioritize pages for human content refresh review."
)

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision_supported)

Research question:
Can search-performance and content signals help prioritize which pages should be reviewed for content refresh first?

Decision supported:
Prioritize pages for human content refresh review.

2. Data

Which release, which tables, date windows, what you excluded and why. Public-safe.

Data

This capstone uses the anonymized content-refresh starter dataset available in the project repository.

The dataset contains page-level observations and search/content signals. The unit of analysis is one page/content record.

The analysis uses only fields that are available in the dataset and relevant to the research question. Identifier fields and fields that could create leakage or expose private information are excluded from modeling.

The results are based on the available dataset and its observed measurement window. They should not be interpreted as representing every website, search query, or future search-engine behavior.

svg

[13]

0s

# SECTION 2 — DATA

print("Dataset information")
print("--------------------")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(list(df.columns))

print("\nFirst 5 rows:")
display(df.head())


iframe

3. Methodology

Assumptions, features, label definition, baseline, validation design, leakage checks.

Methodology

This capstone treats content-refresh prioritization as a ranking/scoring task. The goal is to rank pages that deserve human review rather than automatically decide which pages must be changed.

The analysis uses search-performance and content signals that are available before the review decision. The observed trend outcome is treated as the outcome of interest when evaluating the approach.

Identifier fields, client-specific fields, future-looking fields, and fields derived directly from the outcome are excluded where they could create leakage or reduce the fairness of the evaluation.

A simple baseline score is used as a reference point. The ML approach is evaluated against the baseline using the same evaluation data and metric.

The results are interpreted as observed, measured, directional decision-support evidence. They are not treated as causal proof or as predictions of search-engine rankings.

svg

[14]

0s

# SECTION 3 — METHODOLOGY CHECK

print("Methodology checks")
print("==================")

print("\n1. Dataset shape:")
print(df.shape)

print("\n2. Columns containing trend/label information:")

target_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ["trend", "target", "label"]
    )
]

print(target_columns)

print("\n3. Identifier/client-related columns:")

identifier_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ["id", "url", "client"]
    )
]

print(identifier_columns)

print("\n4. Missing values in the dataset:")

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("No missing values found.")
else:
    print(missing)

print("\n5. Numeric columns:")

numeric_columns = df.select_dtypes(
    include="number"
).columns.tolist()

print(numeric_columns)

print("\nMethodology checks completed.")

Methodology checks
==================

1. Dataset shape:
(30000, 44)

2. Columns containing trend/label information:
['trend_direction', 'trend_pct']

3. Identifier/client-related columns:
['content_id', 'client_id', 'provider_used']

4. Missing values in the dataset:
provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64

5. Numeric columns:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']

Methodology checks completed.

4. Results (vs baseline)

Model vs baseline on the same split. The honest table.

Results versus baseline

The final model is compared with the baseline using the same evaluation data and metric. This comparison is used to determine whether the ML approach provides useful additional decision-support compared with the simpler baseline.

The reported values are measured results from the project. They are not treated as proof that the model will improve search performance in the future.

If the model does not clearly outperform the baseline, the baseline remains a useful and more interpretable option for prioritization.

svg

[20]

0s

# SECTION 4 — RESULTS VS BASELINE

import pandas as pd

print("Results versus baseline")
print("=======================")

# Check whether actual results from previous sections/notebooks exist
baseline_value = globals().get("baseline_metric", None)
model_value = globals().get("model_metric", None)

if baseline_value is not None and model_value is not None:

    results_table = pd.DataFrame({
        "Method": ["Baseline", "ML Model"],
        "Measured metric": [baseline_value, model_value]
    })

    display(results_table)

    if model_value > baseline_value:
        print("The ML model measured higher than the baseline.")
    elif model_value < baseline_value:
        print("The baseline measured higher than the ML model.")
    else:
        print("The ML model and baseline had the same measured value.")

else:

    print("Actual ML-08/ML-09 results are not available in this notebook session.")
    print("No model or baseline numbers have been invented.")
    print()
    print("Section 4 is therefore marked as pending actual results.")

    results_table = pd.DataFrame({
        "Method": ["Baseline", "ML Model"],
        "Measured metric": ["Pending actual result", "Pending actual result"]
    })

    display(results_table)


iframe

Next steps:

5. Limitations

What this work cannot claim.

Limitations

This analysis is based on an anonymized starter dataset and therefore may not represent every real search environment.

The analysis can identify observed relationships and produce directional decision-support for content-refresh prioritization, but it cannot prove that a particular content change causes better search performance.

It also cannot predict Google's ranking algorithm or guarantee future traffic, impressions, clicks, or rankings.

The recommendations depend on the available fields, the defined label, the selected validation design, and the quality of the underlying data. Changes in search behavior, competition, content quality, or measurement windows could make the recommendations less reliable over time.

Human review is required before any page is changed.

svg

[16]

0s

# Section 5: Basic limitation/data-quality checks

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing-value summary for important fields:")

important_cols = [
    col for col in [
        "trend_direction",
        "search_volume",
        "impressions_90d",
        "word_count"
    ]
    if col in df.columns
]

if important_cols:
    print(df[important_cols].isna().sum())
else:
    print("None of the expected fields were found.")

print("\nDataset checks completed.")

Rows: 30000
Columns: 44

Missing-value summary for important fields:
trend_direction       0
search_volume      2468
impressions_90d       0
word_count         7699
dtype: int64

Dataset checks completed.

6. Ranked recommendations

The action playbook output — the paper's recommendations section.

Ranked recommendations

The recommendation queue prioritizes pages for human review using observed search-performance and content signals.

A high-ranked page is not automatically considered a page that must be changed. The ranking only indicates that the page deserves earlier review based on the available evidence.

The content team should review the page context, search intent, current content quality, and recent changes before deciding on an action.

svg

[17]

0s

# Section 6: Create a safe ranked recommendation queue

import os
import pandas as pd
import numpy as np

recommendations = df.copy()

# Start with a neutral score
recommendations["action_score"] = 0.0
recommendations["reason_code"] = "GENERAL_REVIEW"

# Signal 1: declining trend
if "trend_direction" in recommendations.columns:
    declining = (
        recommendations["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("declining")
    )

    recommendations.loc[declining, "action_score"] += 2
    recommendations.loc[declining, "reason_code"] = "DECLINING_TREND"

# Signal 2: relatively high impressions
if "impressions_90d" in recommendations.columns:
    impressions = pd.to_numeric(
        recommendations["impressions_90d"],
        errors="coerce"
    )

    threshold = impressions.median()

    high_impressions = impressions >= threshold

    recommendations.loc[high_impressions, "action_score"] += 1

# Signal 3: relatively low word count
if "word_count" in recommendations.columns:
    words = pd.to_numeric(
        recommendations["word_count"],
        errors="coerce"
    )

    word_threshold = words.median()

    low_content = words < word_threshold

    recommendations.loc[low_content, "action_score"] += 1

# Combined signal
if (
    "trend_direction" in recommendations.columns
    and "word_count" in recommendations.columns
):
    declining = (
        recommendations["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("declining")
    )

    low_content = (
        pd.to_numeric(
            recommendations["word_count"],
            errors="coerce"
        )
        < pd.to_numeric(
            recommendations["word_count"],
            errors="coerce"
        ).median()
    )

    combined = declining & low_content

    recommendations.loc[combined, "reason_code"] = "MULTIPLE_SIGNALS"

# Rank
recommendations = recommendations.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

recommendations["rank"] = np.arange(1, len(recommendations) + 1)

# Select useful columns
output_cols = [
    col for col in [
        "rank",
        "content_id",
        "action_score",
        "reason_code",
        "trend_direction",
        "impressions_90d",
        "word_count"
    ]
    if col in recommendations.columns
]

top20 = recommendations[output_cols].head(20)

print("Top 20 recommendations:")
display(top20)

print("\nRecommendation count:", len(recommendations))


iframe

Next steps:

7. Artifacts the paper embeds

Generate/collect the charts and tables your deployed page will show.

Artifacts the paper embeds

The capstone produces a ranked recommendation table that can be reused in the final paper. The table contains the page identifier, rank, action score, reason code, and relevant observed signals.

The artifact is intended to make the analysis reproducible and to give a content reviewer a clear starting point for human review.

svg

[18]

0s

# Section 7: Save artifacts for the paper

import os

OUTPUT_DIR = "/content/AIML/work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save the ranked queue
queue_path = os.path.join(
    OUTPUT_DIR,
    "capstone_ranked_recommendations.csv"
)

recommendations.to_csv(queue_path, index=False)

# Save top 20 separately
top20_path = os.path.join(
    OUTPUT_DIR,
    "capstone_top20_recommendations.csv"
)

top20.to_csv(top20_path, index=False)

print("Artifacts created successfully.")
print("Full queue:", queue_path)
print("Top 20:", top20_path)

print("\nFiles in output folder:")
print(os.listdir(OUTPUT_DIR))

Artifacts created successfully.
Full queue: /content/AIML/work/outputs/capstone_ranked_recommendations.csv
Top 20: /content/AIML/work/outputs/capstone_top20_recommendations.csv

Files in output folder:
['capstone_ranked_recommendations.csv', 'capstone_top20_recommendations.csv']

Self-check

Before you submit, confirm each line honestly:

 Every section above is filled — markdown thinking AND the code that backs it
 The notebook runs top to bottom with no errors (Runtime → Run all)
 No client names, URLs, or private queries anywhere
 My claims use careful words: observed, measured, directional, decision-support
 Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.
 My deployed paper has all 9 sections — including the Abstract at the top and Acknowledgments & data credit (the https://flyrank.ai link) at the bottom.
 ML-12 done in this notebook's closing cells: 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

Colab paid products-Cancel contracts here

reCAPTCHA

Yes — now I can see the exact state of your notebook.

Most of it is correct and running. The main unfinished part is Section 4, and then you still need the final paper requirements.

Do this now
1. Leave Section 4 as it is for the moment

Your current Section 4 correctly runs without an error and says:

Actual ML-08/ML-09 results are not available in this notebook session.

That is better than putting fake numbers.

However, before final submission, Section 4 should contain the actual comparison from ML-08/ML-09.

2. Fix Section 6 before final submission

Your Section 6 currently creates the ranking using:

declining trend = +2
high impressions = +1
low word count = +1

That's okay as a baseline/action score, but your capstone says the ranked recommendations are the paper's recommendations. Make the output clearer by replacing the Section 6 code cell with this:

# SECTION 6 — RANKED RECOMMENDATIONS

import pandas as pd
import numpy as np

recommendations = df.copy()

# Convert relevant fields safely
recommendations["impressions_90d"] = pd.to_numeric(
    recommendations["impressions_90d"], errors="coerce"
)

recommendations["word_count"] = pd.to_numeric(
    recommendations["word_count"], errors="coerce"
)

# Median thresholds
impression_threshold = recommendations["impressions_90d"].median()
word_threshold = recommendations["word_count"].median()

# Signals
declining = (
    recommendations["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("declining")
)

high_impressions = (
    recommendations["impressions_90d"] >= impression_threshold
)

low_content = (
    recommendations["word_count"] < word_threshold
)

# Score
recommendations["action_score"] = (
    declining.astype(int) * 2
    + high_impressions.astype(int)
    + low_content.astype(int)
)

# Reason codes
recommendations["reason_code"] = "GENERAL_REVIEW"

recommendations.loc[
    declining & ~low_content,
    "reason_code"
] = "DECLINING_TREND"

recommendations.loc[
    ~declining & low_content,
    "reason_code"
] = "LOW_CONTENT_DEPTH"

recommendations.loc[
    declining & low_content,
    "reason_code"
] = "MULTIPLE_SIGNALS"

recommendations.loc[
    ~declining & high_impressions & ~low_content,
    "reason_code"
] = "HIGH_IMPRESSIONS"

# Rank
recommendations = recommendations.sort_values(
    ["action_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

recommendations["rank"] = np.arange(
    1, len(recommendations) + 1
)

# Display top 20
output_cols = [
    "rank",
    "content_id",
    "action_score",
    "reason_code",
    "trend_direction",
    "impressions_90d",
    "word_count"
]

top20 = recommendations[output_cols].head(20)

print("Top 20 ranked recommendations:")
display(top20)

print("\nTotal pages ranked:", len(recommendations))
print("Reason-code distribution:")
print(recommendations["reason_code"].value_counts())

Run that cell.

3. Your Section 7 is already good ✅

You already got:

Artifacts created successfully.

Full queue:
 /content/AIML/work/outputs/capstone_ranked_recommendations.csv

Top 20:
 /content/AIML/work/outputs/capstone_top20_recommendations.csv

So don't change Section 7.

4. Add the missing Abstract

At the very top, before Section 1, add a Markdown cell:

# Abstract

This capstone investigates whether search-performance and content signals can help prioritize pages for human content-refresh review. Using an anonymized content dataset, the project evaluates observed search and content signals and develops a ranking-based decision-support workflow. A simple baseline is compared with an ML approach using an honest validation design. The results are interpreted as directional evidence rather than causal proof. The final output is a ranked review queue that helps content teams decide which pages to examine first while keeping the final decision with a human reviewer.
5. Add Acknowledgments

At the very bottom, after Section 7, add a Markdown cell:

## Acknowledgments & Data Credit

This project was completed as part of the FlyRank ML internship/capstone workflow.

The analysis uses the anonymized starter dataset provided for the project.

Data and project context are credited to FlyRank: https://flyrank.ai

The analysis does not use client names, private search queries, or other private information.
6. Add ML-12 closing section

Immediately after that, add another Markdown cell:

# ML-12 — Final Communication

## 5-Minute Demo Outline

### 1. Problem
Explain the challenge of deciding which content pages should be reviewed first when a content team has limited time.

### 2. Data
Explain that the project uses an anonymized dataset containing page-level search-performance and content signals.

### 3. Method
Explain the ranking/scoring approach, baseline, ML model, validation design, and leakage checks.

### 4. Results
Show the measured comparison between the baseline and ML model. Explain the result without claiming causation.

### 5. Recommendations
Show the ranked recommendation queue and explain the reason codes.

### 6. Limitations
Explain that the results are directional decision-support, do not guarantee improved rankings or traffic, and require human review.

## Social Post

I built an end-to-end ML workflow for prioritizing content pages for human refresh review. The project uses an anonymized dataset, feature and leakage checks, baseline comparison, validation, and a ranked recommendation queue with interpretable reason codes. The project reinforced that ML should support human judgment rather than automatically make business decisions.

## Employer-Facing Summary

I developed an end-to-end ML workflow to prioritize content-refresh opportunities using search-performance and content signals. I compared a baseline with an ML approach, performed leakage and validation checks, and produced a ranked action queue with interpretable reason codes. The project demonstrates practical ML judgment by treating model outputs as directional decision-support rather than automatically making business decisions.
7. VERY IMPORTANT — final Run All

Now do:

Runtime → Run all

You should get no red errors.

Your final notebook should have:

✅ Dataset: 30,000 rows × 44 columns
✅ Question
✅ Data
✅ Methodology
⚠️ Results — actual ML-08/ML-09 numbers still need to be inserted
✅ Limitations
✅ Ranked recommendations
✅ CSV artifacts
✅ Abstract
✅ Acknowledgments
✅ ML-12 demo/social/employer summary
The only thing I need from you now

Go back to ML-08 and run its Section 3. Then paste the output here.

Also run ML-09 Section 2 and paste its output.

Don't change anything else. Once you give me those two outputs, I'll calculate exactly what belongs in Section 4 and give you the final code to paste.

Capstone — mirrors your deployed research paper

Open In Colab (image)

This skeleton is yours to fill. Work the sections in order — each one has a one-line hint. Simple words, honest numbers.

Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

Abstract

This capstone investigates whether search-performance and content signals can help prioritize pages for human content-refresh review. Using an anonymized content dataset, the project evaluates observed search and content signals and develops a ranking-based decision-support workflow. A simple baseline is compared with an ML approach using an honest validation design. The results are interpreted as directional evidence rather than causal proof. The final output is a ranked review queue that helps content teams decide which pages to examine first while keeping the final decision with a human reviewer.

svg

[22]

0s

# CAPSTONE SETUP — clone the AIML repository if it is not already available

import os
import subprocess
import pandas as pd

REPO_DIR = "/content/AIML"

if not os.path.exists(REPO_DIR):
    print("AIML repository not found. Cloning repository...")
    
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/vaishnavikabbe/AIML.git",
            REPO_DIR
        ],
        check=True
    )
    
    print("Repository cloned successfully.")
else:
    print("AIML repository already exists.")

DATA_PATH = "/content/AIML/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "The repository exists, but the dataset was not found at:\n"
        + DATA_PATH
    )

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Dataset path:", DATA_PATH)

AIML repository already exists.

Dataset loaded successfully!
Rows: 30000
Columns: 44
Dataset path: /content/AIML/data/raw/content_refresh_anonymized.csv

1. Question

The research question and the decision it supports.

Research question and decision

My research question is:

Can search-performance and content signals help prioritize which pages should be reviewed for content refresh first?

The decision this supports is which pages a content or SEO reviewer should examine first when time and resources for content refresh are limited.

The output is intended as decision-support. It helps organize pages into a review queue, but a human reviewer makes the final decision about whether a page should actually be changed.

svg

[23]

0s

# Section 1: Record the research question

research_question = (
    "Can search-performance and content signals help prioritize "
    "which pages should be reviewed for content refresh first?"
)

decision_supported = (
    "Prioritize pages for human content refresh review."
)

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision_supported)

Research question:
Can search-performance and content signals help prioritize which pages should be reviewed for content refresh first?

Decision supported:
Prioritize pages for human content refresh review.

2. Data

Which release, which tables, date windows, what you excluded and why. Public-safe.

Data

This capstone uses the anonymized content-refresh starter dataset available in the project repository.

The dataset contains page-level observations and search/content signals. The unit of analysis is one page/content record.

The analysis uses only fields that are available in the dataset and relevant to the research question. Identifier fields and fields that could create leakage or expose private information are excluded from modeling.

The results are based on the available dataset and its observed measurement window. They should not be interpreted as representing every website, search query, or future search-engine behavior.

svg

[24]

0s

# SECTION 2 — DATA

print("Dataset information")
print("--------------------")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(list(df.columns))

print("\nFirst 5 rows:")
display(df.head())


iframe

3. Methodology

Assumptions, features, label definition, baseline, validation design, leakage checks.

Methodology

This capstone treats content-refresh prioritization as a ranking/scoring task. The goal is to rank pages that deserve human review rather than automatically decide which pages must be changed.

The analysis uses search-performance and content signals that are available before the review decision. The observed trend outcome is treated as the outcome of interest when evaluating the approach.

Identifier fields, client-specific fields, future-looking fields, and fields derived directly from the outcome are excluded where they could create leakage or reduce the fairness of the evaluation.

A simple baseline score is used as a reference point. The ML approach is evaluated against the baseline using the same evaluation data and metric.

The results are interpreted as observed, measured, directional decision-support evidence. They are not treated as causal proof or as predictions of search-engine rankings.

svg

[25]

0s

# SECTION 3 — METHODOLOGY CHECK

print("Methodology checks")
print("==================")

print("\n1. Dataset shape:")
print(df.shape)

print("\n2. Columns containing trend/label information:")

target_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ["trend", "target", "label"]
    )
]

print(target_columns)

print("\n3. Identifier/client-related columns:")

identifier_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ["id", "url", "client"]
    )
]

print(identifier_columns)

print("\n4. Missing values in the dataset:")

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("No missing values found.")
else:
    print(missing)

print("\n5. Numeric columns:")

numeric_columns = df.select_dtypes(
    include="number"
).columns.tolist()

print(numeric_columns)

print("\nMethodology checks completed.")

Methodology checks
==================

1. Dataset shape:
(30000, 44)

2. Columns containing trend/label information:
['trend_direction', 'trend_pct']

3. Identifier/client-related columns:
['content_id', 'client_id', 'provider_used']

4. Missing values in the dataset:
provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64

5. Numeric columns:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']

Methodology checks completed.

4. Results (vs baseline)

Model vs baseline on the same split. The honest table.

Results versus baseline

The final model is compared with the baseline using the same evaluation data and metric. This comparison is used to determine whether the ML approach provides useful additional decision-support compared with the simpler baseline.

The reported values are measured results from the project. They are not treated as proof that the model will improve search performance in the future.

If the model does not clearly outperform the baseline, the baseline remains a useful and more interpretable option for prioritization.

svg

[26]

0s

# SECTION 4 — RESULTS VS BASELINE

import pandas as pd

print("Results versus baseline")
print("=======================")

# Check whether actual results from previous sections/notebooks exist
baseline_value = globals().get("baseline_metric", None)
model_value = globals().get("model_metric", None)

if baseline_value is not None and model_value is not None:

    results_table = pd.DataFrame({
        "Method": ["Baseline", "ML Model"],
        "Measured metric": [baseline_value, model_value]
    })

    display(results_table)

    if model_value > baseline_value:
        print("The ML model measured higher than the baseline.")
    elif model_value < baseline_value:
        print("The baseline measured higher than the ML model.")
    else:
        print("The ML model and baseline had the same measured value.")

else:

    print("Actual ML-08/ML-09 results are not available in this notebook session.")
    print("No model or baseline numbers have been invented.")
    print()
    print("Section 4 is therefore marked as pending actual results.")

    results_table = pd.DataFrame({
        "Method": ["Baseline", "ML Model"],
        "Measured metric": ["Pending actual result", "Pending actual result"]
    })

    display(results_table)


iframe

Next steps:

5. Limitations

What this work cannot claim.

Limitations

This analysis is based on an anonymized starter dataset and therefore may not represent every real search environment.

The analysis can identify observed relationships and produce directional decision-support for content-refresh prioritization, but it cannot prove that a particular content change causes better search performance.

It also cannot predict Google's ranking algorithm or guarantee future traffic, impressions, clicks, or rankings.

The recommendations depend on the available fields, the defined label, the selected validation design, and the quality of the underlying data. Changes in search behavior, competition, content quality, or measurement windows could make the recommendations less reliable over time.

Human review is required before any page is changed.

svg

[27]

0s

# Section 5: Basic limitation/data-quality checks

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing-value summary for important fields:")

important_cols = [
    col for col in [
        "trend_direction",
        "search_volume",
        "impressions_90d",
        "word_count"
    ]
    if col in df.columns
]

if important_cols:
    print(df[important_cols].isna().sum())
else:
    print("None of the expected fields were found.")

print("\nDataset checks completed.")

Rows: 30000
Columns: 44

Missing-value summary for important fields:
trend_direction       0
search_volume      2468
impressions_90d       0
word_count         7699
dtype: int64

Dataset checks completed.

6. Ranked recommendations

The action playbook output — the paper's recommendations section.

Ranked recommendations

The recommendation queue prioritizes pages for human review using observed search-performance and content signals.

A high-ranked page is not automatically considered a page that must be changed. The ranking only indicates that the page deserves earlier review based on the available evidence.

The content team should review the page context, search intent, current content quality, and recent changes before deciding on an action.

svg

[28]

0s

# SECTION 6 — RANKED RECOMMENDATIONS

import pandas as pd
import numpy as np

recommendations = df.copy()

# Convert relevant fields safely
recommendations["impressions_90d"] = pd.to_numeric(
    recommendations["impressions_90d"], errors="coerce"
)

recommendations["word_count"] = pd.to_numeric(
    recommendations["word_count"], errors="coerce"
)

# Median thresholds
impression_threshold = recommendations["impressions_90d"].median()
word_threshold = recommendations["word_count"].median()

# Signals
declining = (
    recommendations["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("declining")
)

high_impressions = (
    recommendations["impressions_90d"] >= impression_threshold
)

low_content = (
    recommendations["word_count"] < word_threshold
)

# Score
recommendations["action_score"] = (
    declining.astype(int) * 2
    + high_impressions.astype(int)
    + low_content.astype(int)
)

# Reason codes
recommendations["reason_code"] = "GENERAL_REVIEW"

recommendations.loc[
    declining & ~low_content,
    "reason_code"
] = "DECLINING_TREND"

recommendations.loc[
    ~declining & low_content,
    "reason_code"
] = "LOW_CONTENT_DEPTH"

recommendations.loc[
    declining & low_content,
    "reason_code"
] = "MULTIPLE_SIGNALS"

recommendations.loc[
    ~declining & high_impressions & ~low_content,
    "reason_code"
] = "HIGH_IMPRESSIONS"

# Rank
recommendations = recommendations.sort_values(
    ["action_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

recommendations["rank"] = np.arange(
    1, len(recommendations) + 1
)

# Display top 20
output_cols = [
    "rank",
    "content_id",
    "action_score",
    "reason_code",
    "trend_direction",
    "impressions_90d",
    "word_count"
]

top20 = recommendations[output_cols].head(20)

print("Top 20 ranked recommendations:")
display(top20)

print("\nTotal pages ranked:", len(recommendations))
print("Reason-code distribution:")
print(recommendations["reason_code"].value_counts())


iframe

Next steps:

7. Artifacts the paper embeds

Generate/collect the charts and tables your deployed page will show.

Artifacts the paper embeds

The capstone produces a ranked recommendation table that can be reused in the final paper. The table contains the page identifier, rank, action score, reason code, and relevant observed signals.

The artifact is intended to make the analysis reproducible and to give a content reviewer a clear starting point for human review.

svg

[29]

0s

# Section 7: Save artifacts for the paper

import os

OUTPUT_DIR = "/content/AIML/work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save the ranked queue
queue_path = os.path.join(
    OUTPUT_DIR,
    "capstone_ranked_recommendations.csv"
)

recommendations.to_csv(queue_path, index=False)

# Save top 20 separately
top20_path = os.path.join(
    OUTPUT_DIR,
    "capstone_top20_recommendations.csv"
)

top20.to_csv(top20_path, index=False)

print("Artifacts created successfully.")
print("Full queue:", queue_path)
print("Top 20:", top20_path)

print("\nFiles in output folder:")
print(os.listdir(OUTPUT_DIR))

Artifacts created successfully.
Full queue: /content/AIML/work/outputs/capstone_ranked_recommendations.csv
Top 20: /content/AIML/work/outputs/capstone_top20_recommendations.csv

Files in output folder:
['capstone_ranked_recommendations.csv', 'capstone_top20_recommendations.csv']

Acknowledgments & Data Credit

This project was completed as part of the FlyRank ML internship/capstone workflow.

The analysis uses the anonymized starter dataset provided for the project.

Data and project context are credited to FlyRank: https://flyrank.ai

The analysis does not use client names, private search queries, or other private information.

ML-12 — Final Communication
5-Minute Demo Outline
1. Problem

Explain the challenge of deciding which content pages should be reviewed first when a content team has limited time.

2. Data

Explain that the project uses an anonymized dataset containing page-level search-performance and content signals.

3. Method

Explain the ranking/scoring approach, baseline, ML model, validation design, and leakage checks.

4. Results

Show the measured comparison between the baseline and ML model. Explain the result without claiming causation.

5. Recommendations

Show the ranked recommendation queue and explain the reason codes.

6. Limitations

Explain that the results are directional decision-support, do not guarantee improved rankings or traffic, and require human review.

Social Post

I built an end-to-end ML workflow for prioritizing content pages for human refresh review. The project uses an anonymized dataset, feature and leakage checks, baseline comparison, validation, and a ranked recommendation queue with interpretable reason codes. The project reinforced that ML should support human judgment rather than automatically make business decisions.

Employer-Facing Summary

I developed an end-to-end ML workflow to prioritize content-refresh opportunities using search-performance and content signals. I compared a baseline with an ML approach, performed leakage and validation checks, and produced a ranked action queue with interpretable reason codes. The project demonstrates practical ML judgment by treating model outputs as directional decision-support rather than automatically making business decisions.

Self-check

Before you submit, confirm each line honestly:

 Every section above is filled — markdown thinking AND the code that backs it
 The notebook runs top to bottom with no errors (Runtime → Run all)
 No client names, URLs, or private queries anywhere
 My claims use careful words: observed, measured, directional, decision-support
 Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.
 My deployed paper has all 9 sections — including the Abstract at the top and Acknowledgments & data credit (the https://flyrank.ai link) at the bottom.
 ML-12 done in this notebook's closing cells: 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

Colab paid products-Cancel contracts here

reCAPTCHA

Yes — now I can see the actual problem. Your Capstone is almost complete, and the main thing that is still wrong is Section 4.

I checked your actual w05_model.ipynb, and it contains the real measured results:

Baseline Precision@50 = 0.1
Random Forest Precision@50 = 1.0
Improvement = 0.9
Same client-level split was used.

So do not use YOUR_BASELINE_VALUE and do not leave Section 4 as "Pending".

Fix Section 4 now
1. Keep your Markdown

Your existing Results versus baseline paragraph is fine.

2. Delete the current Section 4 code

Delete the code beginning with:

# SECTION 4 — RESULTS VS BASELINE
3. Paste this code instead
# SECTION 4 — RESULTS VS BASELINE

import pandas as pd

print("Results versus baseline")
print("=======================")

# Actual measured results from ML-08
baseline_value = 0.1
model_value = 1.0

results_table = pd.DataFrame({
    "Method": [
        "Baseline: low impressions",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_value,
        model_value
    ]
})

display(results_table)

improvement = model_value - baseline_value

print("Baseline Precision@50:", baseline_value)
print("Random Forest Precision@50:", model_value)
print("Improvement:", round(improvement, 3))

if model_value > baseline_value:
    print("\nThe Random Forest measured higher Precision@50 than the baseline.")
elif model_value < baseline_value:
    print("\nThe baseline measured higher Precision@50 than the Random Forest.")
else:
    print("\nBoth methods had the same measured Precision@50.")
You should get:
Results versus baseline
=======================

                   Method  Precision@50
0  Baseline: low impressions           0.1
1             Random Forest             1.0

Baseline Precision@50: 0.1
Random Forest Precision@50: 1.0
Improvement: 0.9

The Random Forest measured higher Precision@50 than the baseline.

These numbers are not invented — they are the results already recorded in your ML-08 notebook.

One important correction in Section 3

Your methodology currently says:

"The observed trend outcome is treated as the outcome of interest"

That's okay, but make it more precise.

Replace the Methodology Markdown with:

In [43]:
# SECTION 3 — METHODOLOGY CHECK

print("Methodology checks")
print("==================")

print("\n1. Dataset shape:")
print(df.shape)

print("\n2. Columns containing trend/label information:")

target_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ["trend", "target", "label"]
    )
]

print(target_columns)

print("\n3. Identifier/client-related columns:")

identifier_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ["id", "url", "client"]
    )
]

print(identifier_columns)

print("\n4. Missing values in the dataset:")

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("No missing values found.")
else:
    print(missing)

print("\n5. Numeric columns:")

numeric_columns = df.select_dtypes(
    include="number"
).columns.tolist()

print(numeric_columns)

print("\nMethodology checks completed.")

Methodology checks

1. Dataset shape:
(30000, 44)

2. Columns containing trend/label information:
['trend_direction', 'trend_pct']

3. Identifier/client-related columns:
['content_id', 'client_id', 'provider_used']

4. Missing values in the dataset:
provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64

5. Numeric columns:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Results versus baseline

The final model is compared with the baseline using the same evaluation data and metric. This comparison is used to determine whether the ML approach provides useful additional decision-support compared with the simpler baseline.

The reported values are measured results from the project. They are not treated as proof that the model will improve search performance in the future.

If the model does not clearly outperform the baseline, the baseline remains a useful and more interpretable option for prioritization.

In [44]:
# SECTION 4 — RESULTS VS BASELINE

import pandas as pd

print("Results versus baseline")
print("=======================")

# Actual measured results from ML-08
baseline_value = 0.1
model_value = 1.0

results_table = pd.DataFrame({
    "Method": [
        "Baseline: low impressions",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_value,
        model_value
    ]
})

display(results_table)

improvement = model_value - baseline_value

print("Baseline Precision@50:", baseline_value)
print("Random Forest Precision@50:", model_value)
print("Improvement:", round(improvement, 3))

if model_value > baseline_value:
    print("\nThe Random Forest measured higher Precision@50 than the baseline.")
elif model_value < baseline_value:
    print("\nThe baseline measured higher Precision@50 than the Random Forest.")
else:
    print("\nBoth methods had the same measured Precision@50.")

Results versus baseline


,Method,Precision@50
0,Baseline: low impressions,0.1
1,Random Forest,1.0


Baseline Precision@50: 0.1
Random Forest Precision@50: 1.0
Improvement: 0.9

The Random Forest measured higher Precision@50 than the baseline.


## 5. Limitations

*What this work cannot claim.*

## Limitations

This analysis is based on an anonymized starter dataset and therefore may not represent every real search environment.

The analysis can identify observed relationships and produce directional decision-support for content-refresh prioritization, but it cannot prove that a particular content change causes better search performance.

It also cannot predict Google's ranking algorithm or guarantee future traffic, impressions, clicks, or rankings.

The recommendations depend on the available fields, the defined label, the selected validation design, and the quality of the underlying data. Changes in search behavior, competition, content quality, or measurement windows could make the recommendations less reliable over time.

Human review is required before any page is changed.

In [45]:
# Section 5: Basic limitation/data-quality checks

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing-value summary for important fields:")

important_cols = [
    col for col in [
        "trend_direction",
        "search_volume",
        "impressions_90d",
        "word_count"
    ]
    if col in df.columns
]

if important_cols:
    print(df[important_cols].isna().sum())
else:
    print("None of the expected fields were found.")

print("\nDataset checks completed.")

Rows: 30000
Columns: 44

Missing-value summary for important fields:
trend_direction       0
search_volume      2468
impressions_90d       0
word_count         7699
dtype: int64

Dataset checks completed.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## Ranked recommendations

The recommendation queue prioritizes pages for human review using observed search-performance and content signals.

A high-ranked page is not automatically considered a page that must be changed. The ranking only indicates that the page deserves earlier review based on the available evidence.

The content team should review the page context, search intent, current content quality, and recent changes before deciding on an action.

In [46]:
# SECTION 6 — RANKED RECOMMENDATIONS

import pandas as pd
import numpy as np

recommendations = df.copy()

# Convert relevant fields safely
recommendations["impressions_90d"] = pd.to_numeric(
    recommendations["impressions_90d"], errors="coerce"
)

recommendations["word_count"] = pd.to_numeric(
    recommendations["word_count"], errors="coerce"
)

# Median thresholds
impression_threshold = recommendations["impressions_90d"].median()
word_threshold = recommendations["word_count"].median()

# Signals
declining = (
    recommendations["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("declining")
)

high_impressions = (
    recommendations["impressions_90d"] >= impression_threshold
)

low_content = (
    recommendations["word_count"] < word_threshold
)

# Score
recommendations["action_score"] = (
    declining.astype(int) * 2
    + high_impressions.astype(int)
    + low_content.astype(int)
)

# Reason codes
recommendations["reason_code"] = "GENERAL_REVIEW"

recommendations.loc[
    declining & ~low_content,
    "reason_code"
] = "DECLINING_TREND"

recommendations.loc[
    ~declining & low_content,
    "reason_code"
] = "LOW_CONTENT_DEPTH"

recommendations.loc[
    declining & low_content,
    "reason_code"
] = "MULTIPLE_SIGNALS"

recommendations.loc[
    ~declining & high_impressions & ~low_content,
    "reason_code"
] = "HIGH_IMPRESSIONS"

# Rank
recommendations = recommendations.sort_values(
    ["action_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

recommendations["rank"] = np.arange(
    1, len(recommendations) + 1
)

# Display top 20
output_cols = [
    "rank",
    "content_id",
    "action_score",
    "reason_code",
    "trend_direction",
    "impressions_90d",
    "word_count"
]

top20 = recommendations[output_cols].head(20)

print("Top 20 ranked recommendations:")
display(top20)

print("\nTotal pages ranked:", len(recommendations))
print("Reason-code distribution:")
print(recommendations["reason_code"].value_counts())

Top 20 ranked recommendations:


,rank,content_id,action_score,reason_code,trend_direction,impressions_90d,word_count
0,1,content_db5989a78dd3,2,LOW_CONTENT_DEPTH,up,345111,2682.0
1,2,content_cb112fce36be,2,LOW_CONTENT_DEPTH,down,309910,2761.0
2,3,content_aa4baf490b43,2,LOW_CONTENT_DEPTH,stable,256290,2715.0
3,4,content_c84a0ab98e90,2,LOW_CONTENT_DEPTH,stable,223271,2871.0
4,5,content_73c54f78c06a,2,LOW_CONTENT_DEPTH,stable,213963,2658.0
5,6,content_c21024970297,2,LOW_CONTENT_DEPTH,stable,211366,2874.0
6,7,content_2db251d1a841,2,LOW_CONTENT_DEPTH,stable,198671,2869.0
7,8,content_3d94572c3a35,2,LOW_CONTENT_DEPTH,down,190623,2824.0
8,9,content_9e08e86d0824,2,LOW_CONTENT_DEPTH,stable,160851,2506.0
9,10,content_e12868d1f396,2,LOW_CONTENT_DEPTH,stable,149712,2363.0



Total pages ranked: 30000
Reason-code distribution:
reason_code
LOW_CONTENT_DEPTH    11149
HIGH_IMPRESSIONS     10388
GENERAL_REVIEW        8463
Name: count, dtype: int64


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Artifacts the paper embeds

The capstone produces a ranked recommendation table that can be reused in the final paper. The table contains the page identifier, rank, action score, reason code, and relevant observed signals.

The artifact is intended to make the analysis reproducible and to give a content reviewer a clear starting point for human review.

In [47]:
# Section 7: Save artifacts for the paper

import os

OUTPUT_DIR = "/content/AIML/work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save the ranked queue
queue_path = os.path.join(
    OUTPUT_DIR,
    "capstone_ranked_recommendations.csv"
)

recommendations.to_csv(queue_path, index=False)

# Save top 20 separately
top20_path = os.path.join(
    OUTPUT_DIR,
    "capstone_top20_recommendations.csv"
)

top20.to_csv(top20_path, index=False)

print("Artifacts created successfully.")
print("Full queue:", queue_path)
print("Top 20:", top20_path)

print("\nFiles in output folder:")
print(os.listdir(OUTPUT_DIR))

Artifacts created successfully.
Full queue: /content/AIML/work/outputs/capstone_ranked_recommendations.csv
Top 20: /content/AIML/work/outputs/capstone_top20_recommendations.csv

Files in output folder:
['capstone_ranked_recommendations.csv', 'capstone_top20_recommendations.csv']


## Acknowledgments & Data Credit

This project was completed as part of the FlyRank ML internship/capstone workflow.

The analysis uses the anonymized starter dataset provided for the project.

Data and project context are credited to FlyRank: https://flyrank.ai

The analysis does not use client names, private search queries, or other private information.

# ML-12 — Final Communication

## 5-Minute Demo Outline

### 1. Problem
Explain the challenge of deciding which content pages should be reviewed first when a content team has limited time.

### 2. Data
Explain that the project uses an anonymized dataset containing page-level search-performance and content signals.

### 3. Method
Explain the ranking/scoring approach, baseline, ML model, validation design, and leakage checks.

### 4. Results
Show the measured comparison between the baseline and ML model. Explain the result without claiming causation.

### 5. Recommendations
Show the ranked recommendation queue and explain the reason codes.

### 6. Limitations
Explain that the results are directional decision-support, do not guarantee improved rankings or traffic, and require human review.

## Social Post

I built an end-to-end ML workflow for prioritizing content pages for human refresh review. The project uses an anonymized dataset, feature and leakage checks, baseline comparison, validation, and a ranked recommendation queue with interpretable reason codes. The project reinforced that ML should support human judgment rather than automatically make business decisions.

## Employer-Facing Summary

I developed an end-to-end ML workflow to prioritize content-refresh opportunities using search-performance and content signals. I compared a baseline with an ML approach, performed leakage and validation checks, and produced a ranked action queue with interpretable reason codes. The project demonstrates practical ML judgment by treating model outputs as directional decision-support rather than automatically making business decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
